In [1]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 114.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import onnx
import onnxruntime as ort

import os

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

PyTorch Version: 2.14.1+cpu
GPU Available: False


In [3]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, 3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(16 * 13 * 13, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)

        return x


model = CNN()

print(model)

CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2704, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [4]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.93MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 130kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.46MB/s]

Training Images: 60000
Testing Images: 10000


In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 3

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

print("Training Completed!")

Epoch [1/3], Loss: 0.2908
Epoch [2/3], Loss: 0.1000
Epoch [3/3], Loss: 0.0680
Training Completed!


In [6]:
torch.save(
    model.state_dict(),
    "model_checkpoint.pth"
)

print("Model checkpoint saved!")

Model checkpoint saved!


In [7]:
loaded_model = CNN()

loaded_model.load_state_dict(
    torch.load(
        "model_checkpoint.pth",
        map_location=device,
        weights_only=True
    )
)

loaded_model = loaded_model.to(device)

loaded_model.eval()

print("Checkpoint Loaded Successfully!")

Checkpoint Loaded Successfully!


In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy, "%")

Test Accuracy: 97.95 %


In [9]:
model.eval()

dummy_input = torch.randn(
    1, 1, 28, 28
).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False
)

print("ONNX Model Exported Successfully!")

ONNX Model Exported Successfully!


/tmp/ipykernel_545/49376024.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [10]:
onnx_model = onnx.load("model.onnx")

onnx.checker.check_model(onnx_model)

print("ONNX Model is Valid!")

ONNX Model is Valid!


In [11]:
session = ort.InferenceSession(
    "model.onnx",
    providers=["CPUExecutionProvider"]
)

test_image, test_label = test_dataset[0]

input_data = test_image.unsqueeze(0).numpy()

result = session.run(
    None,
    {"input": input_data}
)

prediction = result[0].argmax()

print("Actual Label:", test_label)
print("ONNX Prediction:", prediction)

Actual Label: 7
ONNX Prediction: 7


In [12]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [14]:
!pip install tensorrt-cu12

In [15]:
import tensorrt as trt

print("TensorRT Version:", trt.__version__)

TensorRT Version: 11.5.0.63


In [19]:
import tensorrt as trt

logger = trt.Logger(trt.Logger.WARNING)

builder = trt.Builder(logger)

# In modern TensorRT (v10+), EXPLICIT_BATCH is the default and the old flag is removed.
network = builder.create_network()

parser = trt.OnnxParser(network, logger)

with open("model.onnx", "rb") as f:
    success = parser.parse(f.read())

if not success:
    for i in range(parser.num_errors):
        print(parser.get_error(i))
    raise RuntimeError("ONNX parsing failed")

config = builder.create_builder_config()

config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    1 << 28
)

# In newer TensorRT versions, flags are defined under trt.BuilderFlag
try:
    config.set_flag(trt.BuilderFlag.FP16)
except AttributeError:
    # Fallback or pass if the flag cannot be set or is configured differently
    print("Could not set FP16 flag directly. Skipping FP16 configuration.")

try:
    engine = builder.build_serialized_network(
        network,
        config
    )
    if engine is not None:
        with open("model.engine", "wb") as f:
            f.write(engine)
        print("TensorRT Engine Created Successfully!")
    else:
        print("TensorRT engine build returned None (GPU is required for building).")
except Exception as e:
    print(f"TensorRT engine build could not complete: {e}")
    print("Note: A CUDA-enabled GPU and appropriate drivers are required to compile TensorRT engines.")

Could not set FP16 flag directly. Skipping FP16 configuration.
TensorRT engine build returned None (GPU is required for building).


In [20]:
import os

files = [
    "model_checkpoint.pth",
    "model.onnx",
    "model.engine"
]

print("MODEL ARTEFACT LIFECYCLE")

for file in files:

    if os.path.exists(file):

        size = os.path.getsize(file) / 1024

        print(file, "-", round(size, 2), "KB")

    else:

        print(file, "- Not Generated")

MODEL ARTEFACT LIFECYCLE
model_checkpoint.pth - 1361.19 KB
model.onnx - 1359.28 KB
model.engine - Not Generated


In [22]:
import os
from google.colab import files

files_to_download = ["model_checkpoint.pth", "model.onnx", "model.engine"]

for file in files_to_download:
    if os.path.exists(file):
        print(f"Downloading {file}...")
        files.download(file)
    else:
        print(f"Skipping {file} (file not generated).")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Skipping model.engine (file not generated).


In [23]:
import torch

checkpoint = torch.load(
    "/content/model_checkpoint.pth",
    map_location="cpu",
    weights_only=True
)

print("Checkpoint loaded successfully!")

print(checkpoint.keys())

Checkpoint loaded successfully!
odict_keys(['conv1.weight', 'conv1.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])


In [24]:
from google.colab import files

uploaded = files.upload()

Saving model_checkpoint (1).pth to model_checkpoint (1).pth


In [25]:
checkpoint = torch.load(
    "model_checkpoint (1).pth",
    map_location="cpu",
    weights_only=True
)

print("Checkpoint loaded successfully!")

Checkpoint loaded successfully!


In [26]:
for name, weights in checkpoint.items():

    print("Layer:", name)

    print("Shape:", weights.shape)

    print("-------------------")

Layer: conv1.weight
Shape: torch.Size([16, 1, 3, 3])
-------------------
Layer: conv1.bias
Shape: torch.Size([16])
-------------------
Layer: fc1.weight
Shape: torch.Size([128, 2704])
-------------------
Layer: fc1.bias
Shape: torch.Size([128])
-------------------
Layer: fc2.weight
Shape: torch.Size([10, 128])
-------------------
Layer: fc2.bias
Shape: torch.Size([10])
-------------------


In [1]:
!nvidia-smi

Mon Nov 23 07:08:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install --upgrade tensorrt-cu13

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.2 MB/s  0:00:00
  Created wheel for tensorrt-cu13: filename=tensorrt_cu13-11.5.0.63-py3-none-any.whl size=23057 sha256=4a4b7efdcfd233eb82b4dc953cc3a16c885c52ca38fd23c92a9a4bd7a9f405a9
  Stored in directory: /root/.cache/pip/wheels/89/4d/e8/f493972a5dc2a1733d7b9f43abd67b965e2356ac4c8657e0fc
  Created wheel for tensorrt_cu13_libs: filename=tensorrt_cu13_libs-11.5.0.63-py3-none-manylinux_2_28_x86_64.whl size=3836415489 sha256=56f4b6764cf86ed22202234b7a89d22a6b817493ab76f0c9d1a8d55d8497efdc
  Stored in directory: /root/.cache/pip/wheels/5e/a8/60/f4680682020406254d287677e3400aaad753f00cee614ceace
Successfully built tensorrt-cu13 tensorrt_cu13_libs
   ━━━━━━━━━━━━━━━━━━

In [3]:
import tensorrt as trt

print("TensorRT Version:", trt.__version__)

logger = trt.Logger(trt.Logger.WARNING)

builder = trt.Builder(logger)

print("TensorRT Installed Successfully!")

TensorRT Version: 11.5.0.63
TensorRT Installed Successfully!


In [4]:
from google.colab import files

uploaded = files.upload()

Saving model (1).onnx to model (1).onnx


In [6]:
import os

target_file = "model (1).onnx"

exists = os.path.exists(target_file)
print(f"ONNX file '{target_file}' exists:", exists)

if exists:
    print("File size:", os.path.getsize(target_file) / 1024, "KB")
else:
    print(f"Error: The file '{target_file}' could not be found.")

ONNX file 'model (1).onnx' exists: True
File size: 1359.275390625 KB


In [9]:
import tensorrt as trt

# Create TensorRT logger
logger = trt.Logger(trt.Logger.WARNING)

# Create builder
builder = trt.Builder(logger)

# Create network (In modern TensorRT 10+, EXPLICIT_BATCH is default and the old flag is removed)
network = builder.create_network()

# Create ONNX parser
parser = trt.OnnxParser(network, logger)

# Read ONNX file
with open("model (1).onnx", "rb") as f:
    success = parser.parse(f.read())

# Check ONNX parsing
if not success:
    print("ONNX parsing failed!")

    for i in range(parser.num_errors):
        print(parser.get_error(i))

    raise RuntimeError("Parsing failed")

print("ONNX file parsed successfully!")

# Create configuration
config = builder.create_builder_config()

config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    1 << 28
)

# Enable FP16 optimization safely
# In newer TensorRT versions, platform_has_fast_fp16 was renamed to platform_has_fp16
has_fp16 = getattr(builder, "platform_has_fp16", False) or getattr(builder, "platform_has_fast_fp16", False)

if has_fp16:
    try:
        config.set_flag(trt.BuilderFlag.FP16)
        print("FP16 optimization enabled!")
    except AttributeError:
        print("FP16 flag could not be set.")
else:
    print("FP16 acceleration is not supported on this platform/GPU.")

# Build TensorRT engine
engine = builder.build_serialized_network(
    network,
    config
)

if engine is None:
    raise RuntimeError("TensorRT engine creation failed!")

# Save engine
with open("model.engine", "wb") as f:
    f.write(engine)

print("TensorRT Engine Created Successfully!")

ONNX file parsed successfully!
FP16 acceleration is not supported on this platform/GPU.
TensorRT Engine Created Successfully!


In [10]:
import os

if os.path.exists("model.engine"):

    print("Engine file exists!")

    print(
        "File size:",
        round(os.path.getsize("model.engine") / 1024, 2),
        "KB"
    )

else:
    print("Engine file not found!")

Engine file exists!
File size: 1406.56 KB


In [11]:
from google.colab import files

files.download("model.engine")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>